## 💥 Apa Itu High Cardinality?

High cardinality terjadi ketika sebuah fitur kategorikal memiliki terlalu banyak nilai unik.<br>

Contoh:
- Fitur kota_pengguna → bisa punya 1.000 kota
- Fitur kode_pos, ID_produk, domain_email, nama_perusahaan → bisa ada ribuan bahkan jutaan nilai unik

### ❗Kenapa High Cardinality Bermasalah?

1. One-Hot Encoding jadi tidak efisien
- Ribuan kolom baru bisa muncul (sparse matrix)
- Konsumsi RAM & waktu komputasi meningkat drastis

2. Overfitting
- Model bisa belajar "menghafal" ID unik, bukan generalisasi pola

3. Model jadi lambat & sulit ditafsirkan

## ⚙️Teknik High Cardinality:

### 1. 🎯 Label Encoding

Mengganti kategori dengan angka.

📌 Kapan dipakai?
- Saat fitur punya urutan alami (misal: level pendidikan)
- Untuk model tree-based (XGBoost, LightGBM) yang bisa menangani angka kategorikal

In [2]:
from sklearn.preprocessing import LabelEncoder

In [6]:
data = ['jakarta', 'bandung', 'surabaya', 'jakarta', 'medan', 'surabaya', 'papua', 'papua']

In [7]:
le = LabelEncoder()
encoded = le.fit_transform(data)

In [8]:
print(encoded)

[1 0 4 1 2 4 3 3]


### 2. 🔲 One-Hot Encoding

**Membuat kolom biner untuk setiap kategori unik.**

📌 Kapan cocok?
- Jika jumlah kategori rendah (< 20)
- Model seperti logistic regression, SVM

In [9]:
import pandas as pd

In [10]:
df = pd.DataFrame({'kota': ['jakarta', 'bandung', 'jakarta', 'surabaya']})

In [11]:
df

,kota
0,jakarta
1,bandung
2,jakarta
3,surabaya


In [13]:
encoded = pd.get_dummies(df, columns=['kota'])
print(encoded)

   kota_bandung  kota_jakarta  kota_surabaya
0         False          True          False
1          True         False          False
2         False          True          False
3         False         False           True


### 3. 🎯 Target Encoding (Mean Encoding)

Gantikan kategori dengan rata-rata target (misalnya rata-rata pembelian per kategori).

📌 Kapan cocok?
- High cardinality + model statistik
- Target berupa numerik atau binary (0/1)
- Cocok untuk data tabular klasifikasi atau regresi

In [15]:
# Contoh data
df = pd.DataFrame({
    'kota': ['jakarta', 'bandung', 'jakarta', 'surabaya', 'bandung'],
    'target': [1, 0, 1, 0, 1]
})

In [16]:
# Hitung rata-rata target per kota
mean_map = df.groupby('kota')['target'].mean()

In [17]:
# Encode
df['kota_encoded'] = df['kota'].map(mean_map)

print(df)

       kota  target  kota_encoded
0   jakarta       1           1.0
1   bandung       0           0.5
2   jakarta       1           1.0
3  surabaya       0           0.0
4   bandung       1           0.5


## 🧠 Rangkuman Pemilihan Teknik:

| Teknik           | Cocok untuk                             | Hindari Saat                         |
| ---------------- | --------------------------------------- | ------------------------------------ |
| Label Encoding   | Tree-based models (XGBoost, CatBoost)   | Model linear tanpa urutan kategori   |
| One-Hot Encoding | < 20 kategori                           | High cardinality                     |
| Target Encoding  | Kategori banyak + target numeric/binary | Jika data kecil (risiko overfitting) |
| Embedding        | Deep learning + kategori ribuan         | Dataset kecil, model klasik          |